In [6]:
import pandas as pd
import os
import openai
from openai.error import RateLimitError, OpenAIError
import tiktoken


OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
openai.api_base = "http://api.openai.com/v1"
openai.proxy = "http://18.139.228.114:81"

In [7]:
xlsx = pd.read_excel("Questions Collection List - by 0818.xlsx")

In [11]:
from time import sleep
from pandas import DataFrame
import functools

GPT3_5 = "gpt-3.5-turbo"
GPT4 = "gpt-4"
REPEAT = 3

def tokenize(model_name: str, text: str) -> str:
    """tokenize the given text"""
    # create a GPT-3.5-Turbo encoder instance
    enc = tiktoken.encoding_for_model(model_name)
    # encode the text using the GPT-3.5-Turbo encoder
    tokenized_text = enc.encode(text)
    return tokenized_text


def count_tokens(text, model):
    return len(tokenize(model, text))


def load_template(file_name):
    with open(file_name, "r", encoding="utf-8") as fp:
        return fp.read()


def paraphrase(df: DataFrame, model_name, repeats):
    results = []
    template = load_template("1_paraphrase.txt")

    model_name = model_name
    for column in df.columns:
        print(f"Column: {column}")
        for text in df[column].dropna():
            messages = [
                {"role": "system", "content": template},
                {"role": "user", "content": text},
            ]
            results.append(text)
            for _ in range(repeats):
                while True:
                    try:
                        response = openai.ChatCompletion.create(
                            model=model_name, messages=messages, temperature=1.0
                        )
                        for choic in response["choices"]:
                            results.append(choic["message"]["content"])
                    except OpenAIError as re:
                        print(re)
                        sleep(60)
                    else:
                        break

    return DataFrame({"Questions": results})


In [12]:
df = paraphrase(xlsx, GPT4, 3)

Column: Adding Funds
Column: Withdrawing Funds
Column: Account Opening
Column: Mutual Funds


In [13]:
df.to_excel("output.xlsx", index=False)